# KARMA – Real-Data Comparison Experiment

Runs `experiments/comparison_realdata.py` on **ETTm1, ETTm2, web_traffic, electricity**.

Methods compared: FO · DynaMask · FIT · WinIT · IG · TimeShap · KARMA · ExtremalMask · TIMING

Metrics: Cell/Lag AUC, Cell/Lag Drop@25%.

> **Before running:** Runtime → Change runtime type → **T4 GPU**

## 0 · Configuration — edit this cell

All paths live here.  Change `DRIVE_ROOT` if your Drive folder has a different name.

In [ ]:
import os, sys
from pathlib import Path

# ── user-configurable ──────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/KARMA_data'   # your Drive folder
ROOT         = '/content/KARMA'                       # where the repo is cloned
DATASETS_RUN = ['ettm1', 'ettm2', 'web_traffic', 'electricity']
TRAIN_EPOCHS = 50          # epochs when training from scratch
DEVICE       = 'auto'      # 'auto' → cuda if available, else cpu
# ──────────────────────────────────────────────────────────────────────────

ROOT         = Path(ROOT)
DRIVE_ROOT   = Path(DRIVE_ROOT)
RAW          = ROOT / 'data' / 'raw'
GEN          = ROOT / 'data' / 'generated'
CKPT_ROOT    = ROOT / 'outputs' / 'checkpoints'
EXP_CKPT_DIR = CKPT_ROOT / 'realdata'
OUT_DIR      = ROOT / 'results' / 'realdata_colab'

DRIVE_RAW    = DRIVE_ROOT / 'raw'
DRIVE_CKPT   = DRIVE_ROOT / 'checkpoints'
DRIVE_OUT    = DRIVE_ROOT / 'results' / 'realdata_colab'

print('ROOT      :', ROOT)
print('DRIVE_ROOT:', DRIVE_ROOT)
print('Datasets  :', DATASETS_RUN)

## 1 · Mount Google Drive

**Expected Drive layout** — create once, reuse across sessions:
```
MyDrive/KARMA_data/
  raw/
    ETTm1.csv  ETTm2.csv
    electricityloaddiagrams/LD2011_2014.txt
    web_traffic/kaggle_web_traffic_dataset_without_missing_values.tsf
  checkpoints/
    ettm1_lstm/best.pt    ettm1_tcn/best.pt
    ettm2_lstm/best.pt    ettm2_tcn/best.pt
    web_traffic_lstm/best.pt   web_traffic_tcn/best.pt
    electricity_lstm/best.pt   electricity_tcn/best.pt
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone KARMA repo and external explainer repos

In [ ]:
import subprocess

def run(cmd, **kw):
    """Run a shell command and raise on failure."""
    result = subprocess.run(cmd, **kw)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {" ".join(str(c) for c in cmd)}')
    return result

KARMA_REPO = 'https://github.com/AmTuTi1999/KARMA-.git'
if not ROOT.exists():
    run(['git', 'clone', '--depth', '1', KARMA_REPO, str(ROOT)])
else:
    print('KARMA already cloned – pulling latest')
    run(['git', '-C', str(ROOT), 'pull'])

EXTERNAL = [
    ('WinIT',         'https://github.com/layer6ai-labs/WinIT.git'),
    ('time_interpret','https://github.com/josephenguehard/time_interpret.git'),
    ('TIMING',        'https://github.com/drumpt/TIMING.git'),
]
for name, url in EXTERNAL:
    dest = ROOT / name
    if not dest.exists():
        print(f'Cloning {name} ...')
        run(['git', 'clone', '--depth', '1', url, str(dest)])
    else:
        print(f'{name}: already present')

## 3 · Install Python dependencies

In [ ]:
%%bash
# setuptools must be pinned first — time_interpret needs pkg_resources
pip install -q "setuptools<81"

# shap>=0.44 supports NumPy 2.x (Colab ships numpy 2.0.2; shap==0.40 crashes on it).
# Pin numpy<2 as a belt-and-suspenders guard for other compiled packages (captum, etc.)
pip install -q "numpy<2"

pip install -q \
    pandas scikit-learn scipy \
    torch \
    captum \
    "shap>=0.41,<0.44" \
    timeshap \
    statsmodels \
    joblib \
    tqdm \
    PyYAML \
    "pytorch-lightning>=2.0" \
    "torchmetrics<1.4" \
    reformer-pytorch \
    tigramite

echo "Done."

In [ ]:
import torch, pytorch_lightning
print('torch           :', torch.__version__)
print('pytorch-lightning:', pytorch_lightning.__version__)
print('GPU             :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4 · Link raw data from Drive

In [ ]:
import shutil

RAW.mkdir(parents=True, exist_ok=True)

def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        print(f'  {dst.name}: already present')
        return
    if not src.exists():
        print(f'  WARNING: {src} not found on Drive!')
        return
    try:
        dst.symlink_to(src)
        print(f'  {dst.name}: symlinked from Drive')
    except OSError:
        (shutil.copytree if src.is_dir() else shutil.copy2)(str(src), str(dst))
        print(f'  {dst.name}: copied from Drive')

for fname in ['ETTm1.csv', 'ETTm2.csv']:
    link_or_copy(DRIVE_RAW / fname, RAW / fname)

link_or_copy(DRIVE_RAW / 'electricityloaddiagrams', RAW / 'electricityloaddiagrams')
link_or_copy(DRIVE_RAW / 'web_traffic',             RAW / 'web_traffic')

## 5 · Preprocess datasets → sliding-window .npy files

In [ ]:
import importlib, sys
sys.path.insert(0, str(ROOT))

PREP_CONFIGS = [
    ('ettm1', 'dataset.ettm1', {
        'CSV_PATH':   str(RAW / 'ETTm1.csv'),
        'OUTPUT_DIR': str(GEN / 'ettm1'),
    }),
    ('ettm2', 'dataset.ettm2', {
        'CSV_PATH':   str(RAW / 'ETTm2.csv'),
        'OUTPUT_DIR': str(GEN / 'ettm2'),
    }),
    ('electricity', 'dataset.electricity', {
        'DATA_PATH':  str(RAW / 'electricityloaddiagrams' / 'LD2011_2014.txt'),
        'OUTPUT_DIR': str(GEN / 'electricity'),
    }),
    ('web_traffic', 'dataset.web_traffic', {
        'DATA_PATH':  str(RAW / 'web_traffic' / 'kaggle_web_traffic_dataset_without_missing_values.tsf'),
        'OUTPUT_DIR': str(GEN / 'web_traffic'),
    }),
]

for ds_name, mod_name, patches in PREP_CONFIGS:
    out = GEN / ds_name
    if (out / 'X_train.npy').exists():
        print(f'{ds_name}: already preprocessed ✓')
        continue
    print(f'\n─── Preprocessing {ds_name} ───')
    mod = importlib.import_module(mod_name)
    for attr, val in patches.items():
        setattr(mod, attr, val)          # patch hard-coded paths
        mod.__dict__[attr] = val         # also patch module dict directly
    mod.main()
    print(f'{ds_name}: done ✓')

## 6 · Link pre-trained checkpoints from Drive

Skip and run §7 instead if you want to train from scratch.

In [ ]:
CKPT_DIRS = [
    'ettm1_lstm', 'ettm1_tcn',
    'ettm2_lstm', 'ettm2_tcn',
    'web_traffic_lstm', 'web_traffic_tcn',
    'electricity_lstm', 'electricity_tcn',
]

for d in CKPT_DIRS:
    src = DRIVE_CKPT / d / 'best.pt'
    dst_dir = CKPT_ROOT / d
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / 'best.pt'
    if dst.exists() or dst.is_symlink():
        print(f'  {d}/best.pt: already present')
    elif src.exists():
        dst.symlink_to(src)
        print(f'  {d}/best.pt: symlinked')
    else:
        print(f'  {d}/best.pt: NOT FOUND on Drive — run §7 to train from scratch')

## 7 · (Optional) Train LSTM + TCN from scratch

Only if §6 reported missing checkpoints.  ~15 min per dataset on T4.

In [ ]:
for ds in DATASETS_RUN:
    for model in ['lstm', 'tcn']:
        ckpt = CKPT_ROOT / f'{ds}_{model}' / 'best.pt'
        if ckpt.exists():
            print(f'{ds}/{model}: checkpoint exists – skipping')
            continue
        print(f'\n─── Training {model.upper()} on {ds} ───')
        r = subprocess.run(
            [sys.executable, '-m', 'pipeline.training_pipeline',
             '--dataset', ds, '--model', model, '--epochs', str(TRAIN_EPOCHS)],
            cwd=str(ROOT)
        )
        if r.returncode != 0:
            print(f'  WARNING: training failed for {ds}/{model}')

## 8 · Pre-flight check

In [ ]:
import json

ok = True
for ds in DATASETS_RUN:
    gen = GEN / ds
    for f in ['X_train.npy', 'X_test.npy', 'metadata.json']:
        if not (gen / f).exists():
            print(f'  MISSING: {gen/f}  ← run §5')
            ok = False

    for arch in ['lstm', 'tcn']:
        ckpt = CKPT_ROOT / f'{ds}_{arch}' / 'best.pt'
        if ckpt.exists():
            # Verify the checkpoint is readable
            import torch
            try:
                state = torch.load(str(ckpt), map_location='cpu', weights_only=True)
                print(f'  {ds}/{arch}: checkpoint ok ({len(state)} tensors)')
            except Exception as e:
                print(f'  CORRUPT checkpoint {ckpt}: {e}')
                ok = False
        else:
            print(f'  MISSING: {ckpt}  ← run §6 or §7')
            ok = False

if ok:
    print('\nAll checks passed ✓  — ready to run experiment')
else:
    print('\n⚠  Fix the issues above before running the experiment')

## 9 · Run attribution comparison experiment

In [ ]:
import torch

EXP_CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE

cmd = [
    sys.executable, '-m', 'experiments.comparison_realdata',
    '--datasets', *DATASETS_RUN,
    '--device',          device,
    '--lag_only',
    '--load_existing',
    '--model_ckpt_dir',  str(CKPT_ROOT),
    '--ckpt_dir',        str(EXP_CKPT_DIR),
    '--out_dir',         str(OUT_DIR),
    '--dynamask_epochs',     '100',
    '--fit_epochs',          '100',
    '--winit_epochs',        '100',
    '--extremalmask_epochs', '300',
]

print('Device :', device)
print('Command:', ' '.join(cmd))

In [ ]:
import subprocess, sys as _sys

# Stream output line-by-line so errors appear immediately in the cell
with subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end='', flush=True)

if proc.returncode != 0:
    raise RuntimeError(f'Experiment failed with exit code {proc.returncode}')
else:
    print('\nExperiment complete ✓')

## 10 · Display results

In [ ]:
import json, pandas as pd
from pathlib import Path

METHODS = ['fo', 'dynamask', 'fit', 'winit', 'ig', 'timeshap', 'karma',
           'extremalmask', 'timing']
ARCHS   = ['lstm', 'tcn']
# The run command above passes --lag_only, so only the lag-family metrics
# (whole-timestep removal) are ever populated — cell-level (feature,time)
# metrics like 'auc'/'drop25' would show up empty. Drop --lag_only from the
# cmd list in §9 if you also want the cell-level AUC/Drop@25% columns.
METRICS = [
    ('lag_auc',    'Lag AUC  ↑'),
    ('lag_drop25', 'Lag Drop@25%  ↑'),
]

result_files = sorted(OUT_DIR.glob('*_results.json'))
all_results  = [json.loads(f.read_text()) for f in result_files]
print(f'Loaded {len(all_results)} result file(s):', [f.stem for f in result_files])

rows = []
for r in all_results:
    ds = r.get('dataset', '?')
    for arch in ARCHS:
        for m in METHODS:
            row = {'dataset': ds, 'arch': arch, 'method': m}
            for key, label in METRICS:
                row[label] = r.get(f'{arch}_{m}_{key}')
            if any(row[lbl] is not None for _, lbl in METRICS):
                rows.append(row)

df = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.4f}'.format)

for _, label in METRICS:
    print(f'\n{"═"*90}')
    print(f'  {label}')
    print('═'*90)
    pivot = (
        df.pivot_table(index=['dataset','arch'], columns='method', values=label, aggfunc='first')
        .reindex(columns=[m for m in METHODS if m in df.method.values])
    )
    display(pivot)

## 11 · Save results to Drive

In [ ]:
import shutil
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for f in result_files:
    shutil.copy2(str(f), str(DRIVE_OUT / f.name))
    print(f'Saved {f.name} → Drive')

# Also save trained checkpoints back to Drive so §6 works next time
for d in [f'{ds}_{arch}' for ds in DATASETS_RUN for arch in ['lstm','tcn']]:
    src = CKPT_ROOT / d / 'best.pt'
    if src.exists() and not src.is_symlink():
        dst_dir = DRIVE_CKPT / d
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(src), str(dst_dir / 'best.pt'))
        print(f'Checkpoint {d}/best.pt → Drive')

print(f'\nAll outputs saved to {DRIVE_OUT}')